In [1]:
from constants import DATA_ROOT_PATH_NAME, BANDPASS, HAMPEL_WINDOW_SIZE, HAMPEL_N_SIGMA, CROP_TMIN, CROP_TMAX, LOCAL_DETREND_WINDOW_SEC, LOCAL_DETREND_STEP_SEC, ASR_CUTOFF, ASR_BLOCKSIZE, ASR_WIN_LEN, ASR_WIN_OVERLAP, ASR_MAX_DROPOUT_FRACTION, ASR_MIN_CLEAN_FRACTION, ASR_MAX_BAD_CHANS

from preprocessing.step.bandpass import BandpassFilterStep
from preprocessing.step.detrend import LocalDetrendStep
from preprocessing.step.hampel import HampelFilterStep
from preprocessing.step.asr import ASRStep
from preprocessing.step.crop import CropStep

from preprocessing.pipeline import PreprocessingPipeline
import numpy as np

from features.factory import FeatureExtractionEngine, FeatureExtractionConfig
from features.categories import FeatureCategory

from eeg.data import EEGRecordedDataProvider

from features.visualization import TopomapFactory



%load_ext autoreload
%autoreload 2

# Création des 2 datasets de features

In [ ]:
recordings = EEGRecordedDataProvider.build(DATA_ROOT_PATH_NAME)

/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Did not find any events.tsv associated with sub-001_task-eyesclosed.

The search_str was "data/sub-001/**/eeg/sub-001*events.tsv"
  raw : mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Unable to map the following column(s) to to MNE:
Gender: F
Age: 57
Group: A
MMSE: 16
  raw : mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Did not find any events.tsv associated with sub-002_task-eyesclosed.

The search_str was "data/sub-002/**/eeg/sub-002*events.tsv"
  raw : mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Unable to map the following column(s) to to MNE:
Gender: F
Age: 78
Group: A
MMSE: 22
  raw : mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:121: RuntimeWarning: Did not fin

In [ ]:
asr_pipeline = PreprocessingPipeline(name="ASR",
                                        steps=[
                                                BandpassFilterStep(BANDPASS),
                                                CropStep(tmin=CROP_TMIN, tmax=CROP_TMAX),
                                                ASRStep(cutoff=ASR_CUTOFF, blocksize=ASR_BLOCKSIZE, win_len=ASR_WIN_LEN, win_overlap=ASR_WIN_OVERLAP, max_dropout_fraction=ASR_MAX_DROPOUT_FRACTION, min_clean_fraction=ASR_MIN_CLEAN_FRACTION, max_bad_chans=ASR_MAX_BAD_CHANS)
                                                ])

dethamp_pipeline = PreprocessingPipeline(name="det-hamp",
                                        steps=[ 
                                                BandpassFilterStep(BANDPASS),
                                                CropStep(tmin=CROP_TMIN, tmax=CROP_TMAX),
                                                LocalDetrendStep(window_sec=LOCAL_DETREND_WINDOW_SEC, step_sec=LOCAL_DETREND_STEP_SEC),
                                                HampelFilterStep(window_size=HAMPEL_WINDOW_SIZE, n_sigma=HAMPEL_N_SIGMA)
                                                ])

In [4]:
recorded_eeg = recordings[14]
asr_processed_eeg = asr_pipeline.compute(recorded_eeg)
dethamp_processed_eeg = dethamp_pipeline.compute(recorded_eeg)

In [ ]:
categories_to_extract = [FeatureCategory.WAVELET, FeatureCategory.TEMPORAL, FeatureCategory.POWER_RATIO, FeatureCategory.SPECTRAL]
config = FeatureExtractionConfig(categories_to_extract=categories_to_extract, wamp_threshold=10e-9, ppc_epoch_duration=2)
feature_extraction_engine = FeatureExtractionEngine(config=config)

dethamp_processed_extraction_result = feature_extraction_engine.extract(dethamp_processed_eeg)


asr_processed_extraction_result = feature_extraction_engine.extract(asr_processed_eeg)


/mnt/ssd2/pth-eeg/eeg/eeg/ppc.py:395: RuntimeWarning: fmin=1.000 Hz corresponds to 2.000 < 5 cycles based on the epoch length 2.000 sec, need at least 5.000 sec epochs or fmin=2.500. Spectrum estimate will be unreliable.
  conn: Connectivity = spectral_connectivity_epochs(
/mnt/ssd2/pth-eeg/eeg/eeg/ppc.py:395: RuntimeWarning: fmin=1.000 Hz corresponds to 2.000 < 5 cycles based on the epoch length 2.000 sec, need at least 5.000 sec epochs or fmin=2.500. Spectrum estimate will be unreliable.
  conn: Connectivity = spectral_connectivity_epochs(


# Import de toutes les features de tous les participants

In [2]:
from features.io import FeaturesDatasetIO
dethamp_dataset = FeaturesDatasetIO.load("computed_data/dethamp")
asr_dataset = FeaturesDatasetIO.load("computed_data/asr")

In [3]:
dethamp_dataset.feature_names

['theta_beta_ratio',
 'theta_alpha_ratio',
 'gamma_alpha_ratio',
 'spectral_power_ratio',
 'alpha_dominant_frequency',
 'gamma_dominant_frequency',
 'spectral_rolloff',
 'spectral_centroid',
 'spectral_spread',
 'spectral_flux',
 'spectral_skewness',
 'spectral_kurtosis',
 'variance',
 'skewness',
 'kurtosis',
 'peak_amplitude',
 'shape_factor',
 'impulse_factor',
 'crest_factor',
 'clearance_factor',
 'willison_amplitude',
 'zero_crossing_rate',
 'wavelet_energy_approximate',
 'wavelet_energy_detail',
 'relative_wavelet_energy',
 'wavelet_packet_energy_approximate',
 'wavelet_packet_energy_detail',
 'relative_wavelet_packet_energy']

In [4]:
dethamp_dataset.to_long_dataframe()

,channel,feature,value,subject_id,subject_age,subject_health,subject_mmse
0,Fp1,theta_beta_ratio,0.008586,082,63,Frontotemporal Dementia,27
1,Fp2,theta_beta_ratio,0.009132,082,63,Frontotemporal Dementia,27
2,F3,theta_beta_ratio,0.007954,082,63,Frontotemporal Dementia,27
3,F4,theta_beta_ratio,0.011192,082,63,Frontotemporal Dementia,27
4,C3,theta_beta_ratio,0.008844,082,63,Frontotemporal Dementia,27
...,...,...,...,...,...,...,...
46811,T5,relative_wavelet_packet_energy,0.933886,022,68,Alzheimer,20
46812,T6,relative_wavelet_packet_energy,0.932540,022,68,Alzheimer,20
46813,Fz,relative_wavelet_packet_energy,0.938957,022,68,Alzheimer,20
46814,Cz,relative_wavelet_packet_energy,0.935803,022,68,Alzheimer,20


# Tests statistiques

In [38]:
from stats.queries import QueryFactoryConfig, QueryFactory
from stats.runner import StatisticalTestRunner
from stats.correction.fdr import FDRCorrector

factory = QueryFactory(
    QueryFactoryConfig.from_lists(
        subject_variables={"subject_age", "subject_mmse", "subject_id", "subject_mmse", "subject_health"},
        eeg_features=dethamp_dataset.feature_names,
    )
)

query = factory.compare_groups(
    target="shape_factor",
    group_col="subject_health",
    group_a="Healthy",
    group_b="Alzheimer",
    test_kind="wilcoxon_rank_sum",
    scope="all_channels"
)



raw_result_set = StatisticalTestRunner.run(query, dethamp_dataset)
corrected_result_set = FDRCorrector.correct(raw_result_set)

corrected_result_set.to_dataframe()

,key,target,statistic,p_value,p_value_corrected,correction_method,alpha,reject_null,test_name,n_x,n_y,x_name,y_name,label
0,Fp1,shape_factor,-2.098290,0.035880,0.219890,fdr_bh,0.05,False,wilcoxon-rank-sum,29,36,"shape_factor (Healthy, Fp1)","shape_factor (Alzheimer, Fp1)","shape_factor (Healthy, Fp1) vs shape_factor (A..."
1,Fp2,shape_factor,-1.623206,0.104545,0.248295,fdr_bh,0.05,False,wilcoxon-rank-sum,29,36,"shape_factor (Healthy, Fp2)","shape_factor (Alzheimer, Fp2)","shape_factor (Healthy, Fp2) vs shape_factor (A..."
2,F3,shape_factor,-1.992716,0.046293,0.219890,fdr_bh,0.05,False,wilcoxon-rank-sum,29,36,"shape_factor (Healthy, F3)","shape_factor (Alzheimer, F3)","shape_factor (Healthy, F3) vs shape_factor (Al..."
3,F4,shape_factor,-2.401816,0.016314,0.178813,fdr_bh,0.05,False,wilcoxon-rank-sum,29,36,"shape_factor (Healthy, F4)","shape_factor (Alzheimer, F4)","shape_factor (Healthy, F4) vs shape_factor (Al..."
4,C3,shape_factor,-1.438450,0.150306,0.315115,fdr_bh,0.05,False,wilcoxon-rank-sum,29,36,"shape_factor (Healthy, C3)","shape_factor (Alzheimer, C3)","shape_factor (Healthy, C3) vs shape_factor (Al..."
5,C4,shape_factor,-1.741977,0.081513,0.221248,fdr_bh,0.05,False,wilcoxon-rank-sum,29,36,"shape_factor (Healthy, C4)","shape_factor (Alzheimer, C4)","shape_factor (Healthy, C4) vs shape_factor (Al..."
6,P3,shape_factor,0.026394,0.978943,0.978943,fdr_bh,0.05,False,wilcoxon-rank-sum,29,36,"shape_factor (Healthy, P3)","shape_factor (Alzheimer, P3)","shape_factor (Healthy, P3) vs shape_factor (Al..."
7,P4,shape_factor,-1.121727,0.261978,0.451403,fdr_bh,0.05,False,wilcoxon-rank-sum,29,36,"shape_factor (Healthy, P4)","shape_factor (Alzheimer, P4)","shape_factor (Healthy, P4) vs shape_factor (Al..."
8,O1,shape_factor,-0.395904,0.692176,0.773608,fdr_bh,0.05,False,wilcoxon-rank-sum,29,36,"shape_factor (Healthy, O1)","shape_factor (Alzheimer, O1)","shape_factor (Healthy, O1) vs shape_factor (Al..."
9,O2,shape_factor,-0.567462,0.570400,0.677350,fdr_bh,0.05,False,wilcoxon-rank-sum,29,36,"shape_factor (Healthy, O2)","shape_factor (Alzheimer, O2)","shape_factor (Healthy, O2) vs shape_factor (Al..."
